In [1]:
import numpy as np 
import illustris_python as il

In [ ]:
def Filter(basePath, snapNum, h=0.6774, 
           log_mstar_bounds=None, 
           log_mbh_bounds=None, 
           sfr_bounds=None, 
           log_ssfr_bounds=None, 
           log_mhalo_bounds=None):

    
    # 1. 动态确定需要加载的 Halo 字段
    halo_fields = ['GroupFirstSub']
    if log_mhalo_bounds is not None:
        halo_fields.append('GroupMass')
        
    halos = il.groupcat.loadHalos(basePath, snapNum, fields=halo_fields)
    
    # 统一将返回结果包装为字典
    if not isinstance(halos, dict):
        halos = {halo_fields[0]: halos}
    
    valid_halo_mask = halos['GroupFirstSub'] != -1
    subhalo_ids = halos['GroupFirstSub'][valid_halo_mask]
    
    final_mask = np.ones(len(subhalo_ids), dtype=bool)
    
    # --- 主晕层级筛选 ---
    if log_mhalo_bounds is not None:
        mhalo_sim = halos['GroupMass'][valid_halo_mask]
        min_sim = (10**log_mhalo_bounds[0]) * h / 1e10
        max_sim = (10**log_mhalo_bounds[1]) * h / 1e10
        final_mask &= (mhalo_sim >= min_sim) & (mhalo_sim <= max_sim)

    # 2. 动态确定需要加载的 Subhalo 字段
    subhalo_fields = set()
    if log_mstar_bounds or log_ssfr_bounds:
        subhalo_fields.add('SubhaloMassType')
    if log_mbh_bounds:
        subhalo_fields.add('SubhaloBHMass')
    if sfr_bounds or log_ssfr_bounds:
        subhalo_fields.add('SubhaloSFR')
        
    if not subhalo_fields:
        return subhalo_ids[final_mask]
        
    subhalo_fields_list = list(subhalo_fields)
    subhalos = il.groupcat.loadSubhalos(basePath, snapNum, fields=subhalo_fields_list)
    
    # 统一将返回结果包装为字典
    if not isinstance(subhalos, dict):
        subhalos = {subhalo_fields_list[0]: subhalos}
    
    # --- 子晕层级筛选 ---
    if log_mstar_bounds is not None:
        mstar_sim = subhalos['SubhaloMassType'][subhalo_ids, 4]
        min_sim = (10**log_mstar_bounds[0]) * h / 1e10
        max_sim = (10**log_mstar_bounds[1]) * h / 1e10
        final_mask &= (mstar_sim >= min_sim) & (mstar_sim <= max_sim)
        
    if log_mbh_bounds is not None:
        mbh_sim = subhalos['SubhaloBHMass'][subhalo_ids]
        min_sim = (10**log_mbh_bounds[0]) * h / 1e10
        max_sim = (10**log_mbh_bounds[1]) * h / 1e10
        final_mask &= (mbh_sim >= min_sim) & (mbh_sim <= max_sim)
        
    if sfr_bounds is not None:
        sfr = subhalos['SubhaloSFR'][subhalo_ids]
        final_mask &= (sfr >= sfr_bounds[0]) & (sfr <= sfr_bounds[1])
        
    if log_ssfr_bounds is not None:
        mstar_physical = subhalos['SubhaloMassType'][subhalo_ids, 4] * 1e10 / h
        sfr = subhalos['SubhaloSFR'][subhalo_ids]
        
        nonzero_mask = (mstar_physical > 0) & (sfr > 0)
        ssfr = np.zeros_like(sfr)
        ssfr[nonzero_mask] = sfr[nonzero_mask] / mstar_physical[nonzero_mask]
        
        log_ssfr = np.full_like(ssfr, -np.inf)
        log_ssfr[nonzero_mask] = np.log10(ssfr[nonzero_mask])
        
        final_mask &= (log_ssfr >= log_ssfr_bounds[0]) & (log_ssfr <= log_ssfr_bounds[1])

    return subhalo_ids[final_mask]

In [18]:
basePath = '/public/home/zju_visitor/LiuYuanhao/tng_data/output' 
snapNum = 33

subhalo_ids = Filter(basePath, snapNum, h=0.6774, 
              log_mstar_bounds=(10.0, 11.0), 
              log_mbh_bounds=None, 
              sfr_bounds=None, 
              log_ssfr_bounds=(-10, -9), 
              log_mhalo_bounds=None)

print(f"共提取 {len(subhalo_ids)} 个测试中心星系")

共提取 194 个测试中心星系


In [2]:
subhalo_ids = [ 59075,  60750,  71129,  72910,  75667 ]

In [4]:
import os
import re          
import traceback
from API_methods import load_catalogs, analyze_cold_streams_pipeline

# =========================================================================
# Global configuration
# =========================================================================
basePath = '/public/home/zju_visitor/LiuYuanhao/tng_data/output' 
snapNum = 33

cutout_dir = '/public/home/zju_visitor/LiuYuanhao/cold_stream/simulation_results/cutouts'
output_dir = '/public/home/zju_visitor/LiuYuanhao/cold_stream/simulation_results/streams'

# Complete pipeline parameters with all required arguments
PIPELINE_KWARGS = {
    'rmin_fac': 0.15,              # Inner radius fraction of R200c
    'base_edge_tolerance': 1.25,   # Base edge tolerance for connectivity
    'min_edge_tolerance': 1.05,    # Minimum edge tolerance
    'max_edge_tolerance': 1.45,    # Maximum edge tolerance
    'merge_fraction': 0.01,        # KDTree merge tolerance as fraction of R200c
    'thresh_ab': 1.5,              # PCA major/minor axis ratio threshold
    'thresh_ac': 3.0               # PCA major/short axis ratio threshold
}

# =========================================================================
# Utility functions
# =========================================================================
def get_target_subhalo_ids(cutout_dir, snapNum):
    """
    Scan cutout directory and extract subhalo IDs for current snapshot
    """
    if not os.path.exists(cutout_dir):
        print(f"[Warning] Cutout directory does not exist: {cutout_dir}")
        return []
    
    subhalo_ids = []
    pattern = re.compile(rf"cutout_snap{snapNum:03d}_sh(\d+)\.hdf5$")
    
    try:
        for filename in os.listdir(cutout_dir):
            match = pattern.search(filename)
            if match:
                try:
                    subhalo_id = int(match.group(1))
                    subhalo_ids.append(subhalo_id)
                except (ValueError, TypeError):
                    continue
                    
        return sorted(subhalo_ids)
    except Exception as e:
        print(f"[Error] Failed to scan directory {cutout_dir}: {e}")
        traceback.print_exc()
        return []

def run_single_subhalo(subhalo_id, snapNum, catalogs, cutout_dir, output_dir, **kwargs):
    """
    Process single subhalo for cold stream detection
    Returns: (success_status, stream_count)
    """
    target_file = os.path.join(cutout_dir, f"cutout_snap{snapNum:03d}_sh{subhalo_id}.hdf5")
    
    if not os.path.exists(target_file):
        print(f"[Warning] Cutout file not found: {target_file}, skipping subhalo {subhalo_id}")
        return False, 0

    print(f"\n>>> Processing subhalo {subhalo_id}...")
    
    try:
        # Execute pipeline with all required parameters
        objects, props = analyze_cold_streams_pipeline(
            subhalo_id=subhalo_id,
            snapNum=snapNum,
            catalogs=catalogs,
            cutout_dir=cutout_dir,
            output_dir=output_dir,
            **kwargs
        )

        # Extract stream count safely
        stream_count = objects.get('count', 0) if isinstance(objects, dict) else 0
        
        if stream_count > 0:
            print(f"[Success] Subhalo {subhalo_id} detected {stream_count} cold streams")
            print(f"[Info] Results saved to: {output_dir}")
        else:
            print(f"[Info] Subhalo {subhalo_id} processed successfully, no cold streams detected")
            
        return True, stream_count
            
    except Exception as e:
        print(f"[Error] Failed to process subhalo {subhalo_id}: {str(e)}")
        traceback.print_exc()
        return False, 0

# =========================================================================
# Main execution
# =========================================================================
def main():
    print(f"Loading global catalogs for snapshot {snapNum}...")
    try:
        catalogs = load_catalogs(basePath, snapNum)
        print("Global catalogs loaded successfully")
        
        # Basic validation of loaded catalogs
        required_keys = ['grnrs', 'r200c', 'm_vir', 'sub_pos', 'sub_vel', 'sub_rad', 'a', 'BoxSize']
        for key in required_keys:
            if key not in catalogs:
                raise ValueError(f"Missing required catalog field: {key}")
                
        print(f"Box size: {catalogs['BoxSize']} kpc/h, Scale factor: {catalogs['a']:.4f}")
        
    except Exception as e:
        print(f"[Critical Error] Failed to load catalogs: {str(e)}")
        traceback.print_exc()
        return

    # Get target subhalo IDs
    subhalo_ids = get_target_subhalo_ids(cutout_dir, snapNum)
    total = len(subhalo_ids)
    
    if total == 0:
        print(f"\n[Info] No cutout files found for snapshot {snapNum} in {cutout_dir}")
        print("Please verify:")
        print(f"1. Cutout files exist with pattern: cutout_snap{snapNum:03d}_sh*.hdf5")
        print(f"2. Directory path is correct: {cutout_dir}")
        return

    print(f"\n=== Starting batch processing of {total} subhalos ===")
    
    # Track processing statistics
    success_count = 0
    stream_detection_count = 0
    failed_subhalos = []
    
    for index, subhalo_id in enumerate(subhalo_ids, 1):
        print(f"\n--- Progress: {index}/{total} (Subhalo ID: {subhalo_id}) ---")
        success, stream_count = run_single_subhalo(
            subhalo_id=subhalo_id,
            snapNum=snapNum,
            catalogs=catalogs,
            cutout_dir=cutout_dir,
            output_dir=output_dir,
            **PIPELINE_KWARGS
        )
        
        if success:
            success_count += 1
            if stream_count > 0:
                stream_detection_count += 1
        else:
            failed_subhalos.append(subhalo_id)

    # Print summary statistics
    print("\n" + "="*60)
    print("PROCESSING SUMMARY")
    print("="*60)
    print(f"Total subhalos processed: {total}")
    print(f"Successfully processed: {success_count}")
    print(f"Failed processing: {len(failed_subhalos)}")
    print(f"Subhalos with detected cold streams: {stream_detection_count}")
    print(f"Success rate: {success_count/total*100:.1f}%")
    
    if failed_subhalos:
        print(f"\nFailed subhalo IDs: {failed_subhalos}")
    
    print("\n=== All cold stream detection tasks completed ===")

if __name__ == "__main__":
    main()

Loading global catalogs for snapshot 33...
Global catalogs loaded successfully
Box size: 35000.0 kpc/h, Scale factor: 0.3331

=== Starting batch processing of 5 subhalos ===

--- Progress: 1/5 (Subhalo ID: 59075) ---

>>> Processing subhalo 59075...

Starting cold stream analysis | Snapshot: 33 | Subhalo: 59075
Removed 1636 satellite galaxies and 675758 gas cells
Computing physical properties for subhalo [59075]
[59075] Physical filtering complete: retained 261991 cold stream cells (V_vir=346.26 km/s)
[59075] Halo mass M_vir = 3.21e+02, adaptive edge tolerance: 1.450
Performing cold stream topology reconstruction
Processing 261991 cold stream cells with adaptive edge tolerance: 1.450
Topology construction complete: 15.633s (nodes: 261991, edges: 1384653)
Topology reconstruction complete: identified 4335 independent stream components
BFS traversal time: 0.0749s. Total module time: 15.717s
[59075] Merging fragmented streams, initial components: 4335
Merge tolerance: 3.46 kpc/h (1.0% R200

KeyboardInterrupt: 

In [ ]:
from API_methods import load_catalogs, periodic_displacement

def calculate_single_rmax(file_path, subhalo_id, catalogs):
    """计算单个星系 Cutout 的严格物理 r_max"""
    parent_halo_id = catalogs['grnrs'][subhalo_id]
    r200c = catalogs['r200c'][parent_halo_id]
    center_pos = catalogs['sub_pos'][subhalo_id]
    BoxSize = catalogs['BoxSize']
    
    with h5py.File(file_path, 'r') as f:
        if 'PartType0' not in f or 'Coordinates' not in f['PartType0']:
            return r200c, 0.0, 0.0
        pos = f['PartType0/Coordinates'][()]
        
    if len(pos) == 0:
        return r200c, 0.0, 0.0
        
    dx = periodic_displacement(pos, center_pos, BoxSize)
    L_x = dx[:, 0].max() - dx[:, 0].min()
    L_y = dx[:, 1].max() - dx[:, 1].min()
    L_z = dx[:, 2].max() - dx[:, 2].min()

    ratio_x = (L_x / 2) / r200c
    ratio_y = (L_y / 2) / r200c
    ratio_z = (L_z / 2) / r200c
    
    min_ratio = min(ratio_x, ratio_y, ratio_z)
    r_max = min_ratio * r200c 
    
    return r200c, r_max, min_ratio

def check_specific_subhalos(snapNum, basePath, cutout_dir, target_ids):
    """仅检测指定的星系列表，并在终端打印结果"""
    print(f"正在加载快照 {snapNum} 的全局星表...")
    catalogs = load_catalogs(basePath, snapNum)
    
    print(f"\n开始检测指定的 {len(target_ids)} 个星系 Cutout...\n")
    print("-" * 65)
    print(f"{'Subhalo ID':<10} | {'R200c (kpc/h)':<15} | {'R_max (kpc/h)':<15} | {'覆盖倍数':<10}")
    print("-" * 65)
    
    ratios_info = [] 
    
    for subhalo_id in target_ids:
        # 拼接文件名
        file_path = os.path.join(cutout_dir, f"cutout_snap{snapNum:03d}_sh{subhalo_id}.hdf5")
        
        # 检查文件是否存在
        if not os.path.exists(file_path):
            print(f"{subhalo_id:<10} | {'[文件缺失]':<15} | {'N/A':<15} | N/A")
            continue
            
        r200c, r_max, ratio = calculate_single_rmax(file_path, subhalo_id, catalogs)
        print(f"{subhalo_id:<10} | {r200c:<15.2f} | {r_max:<15.2f} | {ratio:.2f} R200c")
        
        if ratio > 0:
            ratios_info.append((subhalo_id, ratio))

    # 打印最终统计摘要
    if ratios_info:
        ratios_only = [r[1] for r in ratios_info]
        avg_ratio = sum(ratios_only) / len(ratios_only)
        min_info = min(ratios_info, key=lambda x: x[1])
        max_info = max(ratios_info, key=lambda x: x[1])
        
        print("-" * 65)
        print("【统计摘要】")
        print(f"成功读取星系: {len(ratios_info)} 个")
        print(f"平均覆盖倍数: {avg_ratio:.2f} R200c")
        print(f"最小覆盖倍数: {min_info[1]:.2f} R200c  (Subhalo {min_info[0]})")
        print(f"最大覆盖倍数: {max_info[1]:.2f} R200c  (Subhalo {max_info[0]})")
        print("=" * 65)

if __name__ == "__main__":
    basePath = '/public/home/zju_visitor/LiuYuanhao/tng_data/output' 
    cutout_dir = '/public/home/zju_visitor/LiuYuanhao/cold_stream/simulation_results/cutouts'
    snapNum = 33
    
    # 你指定的星系列表
    subhalo_ids = [238669, 649346]
    
    check_specific_subhalos(snapNum, basePath, cutout_dir, subhalo_ids)